In [1]:
# 환경 변수 로드
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="./env/.env")

api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# Chat 모델 및 프롬프트 설정
from langchain.chat_models import ChatOpenAI
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferWindowMemory

llm = ChatOpenAI(
    temperature= 0.1
)   

memory = ConversationBufferWindowMemory(
    k = 4,
    memory_key="chat_history",
    return_messages=True # 메시지 객체(리스트) 형태로 결과를 반환하도록 설정
)


def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})


# Few-shot 예시 데이터
examples = [
    {
        "movie": "탑건",
        "answer": "🛩️👨‍✈️🔥",
    },
    {
        "movie": "대부",
        "answer": "👨‍👨‍👦🔫🍝",
    },
    {
        "movie": "조커",
        "answer": "🤡😂🔫",
    },
]

# 예시들을 담을 메시지 포맷
example_format = ChatPromptTemplate.from_messages(
    [
        ("human", "{movie}"),
        ("ai", "{answer}")
    ]
)

few_shot_template = FewShotChatMessagePromptTemplate(
    example_prompt = example_format, 
    examples = examples,
)
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 영화 전문가야."), # 시스템 역할 정의
    MessagesPlaceholder(variable_name="chat_history"),
    few_shot_template, # 위에서 만든 예시 데이터 삽입
    ("human", "{question}") # 실제 사용자 질문
])


chain = final_prompt | llm
response1 = chain.invoke({
    "question": "알라딘",
    "chat_history": get_history()["chat_history"]
})
add_message("알라딘", response1.content)
print(response1.content)


response2 = chain.invoke({
    "question": "해리포터와 마법사의 돌",
    "chat_history": get_history()["chat_history"]
})
add_message("해리포터와 마법사의 돌", response2.content)
print(response2.content)


response3 = chain.invoke({
    "question": "내가 처음 질문한 영화가 뭐야?",
    "chat_history": get_history()["chat_history"]
})

print(response3.content)

🧞‍♂️🕌🌟
🧙‍♂️⚡🔮
네가 처음 질문한 영화는 '알라딘'이었어요!
